# HEN Standalone Final Experiments Notebook

This is a standalone English notebook for the final Flat CNN vs HEN comparison. It does **not** clone any repository and does **not** import project files. The hierarchy, optional data download, model definitions, training loops, experiment runners, and final result comparison are included directly in this `.ipynb` file.

Full training can take hours, so long training is disabled by default. The final measured local results are embedded as notebook outputs, while the code below allows a reader to reproduce selected runs when they choose to.

## 1. Environment setup

Run the optional dependency cell in a fresh Colab runtime. The notebook itself is self-contained; the only external resources are Python packages and, if enabled, the Hugging Face ImageNet mirror used to download images.

In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    MobileNet_V3_Large_Weights,
    MobileNet_V3_Small_Weights,
    ResNet18_Weights,
    ShuffleNet_V2_X1_0_Weights,
    convnext_tiny,
    mobilenet_v3_large,
    mobilenet_v3_small,
    resnet18,
    shufflenet_v2_x1_0,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_LONG_TRAINING = False
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Standalone notebook ready. No repository clone is required.")
print("Device:", DEVICE)
print("RUN_LONG_TRAINING:", RUN_LONG_TRAINING)

Standalone notebook ready. No repository clone is required.
Device: cuda
RUN_LONG_TRAINING: False


In [ ]:
# Optional dependency cell for fresh Colab runtimes.
# Colab usually already has torch/torchvision installed.
!pip install -q pandas matplotlib datasets huggingface_hub pyarrow fsspec

## 2. The 3-9-27 hierarchy

The final experiments use a balanced 27-class ImageNet subset arranged as a three-level tree: 3 coarse groups, 9 mid-level groups, and 27 leaf classes.

In [2]:
TARGET_CLASSES = [
    (11, "n01531178", "goldfinch", "animal", "bird"),
    (15, "n01558993", "robin", "animal", "bird"),
    (22, "n01614925", "bald_eagle", "animal", "bird"),
    (207, "n02099601", "golden_retriever", "animal", "canine"),
    (235, "n02106662", "german_shepherd", "animal", "canine"),
    (236, "n02107142", "doberman", "animal", "canine"),
    (283, "n02123394", "persian_cat", "animal", "feline"),
    (284, "n02123597", "siamese_cat", "animal", "feline"),
    (285, "n02124075", "egyptian_cat", "animal", "feline"),
    (404, "n02690373", "airliner", "vehicle", "aircraft"),
    (405, "n02692877", "airship", "vehicle", "aircraft"),
    (895, "n04552348", "warplane", "vehicle", "aircraft"),
    (407, "n02701002", "ambulance", "vehicle", "road_vehicle"),
    (555, "n03345487", "fire_engine", "vehicle", "road_vehicle"),
    (817, "n04285008", "sports_car", "vehicle", "road_vehicle"),
    (472, "n02951358", "canoe", "vehicle", "watercraft"),
    (510, "n03095699", "container_ship", "vehicle", "watercraft"),
    (814, "n04273569", "speedboat", "vehicle", "watercraft"),
    (951, "n07749582", "lemon", "food", "fruit"),
    (954, "n07753592", "banana", "food", "fruit"),
    (957, "n07768694", "pomegranate", "food", "fruit"),
    (937, "n07714990", "broccoli", "food", "vegetable"),
    (938, "n07715103", "cauliflower", "food", "vegetable"),
    (943, "n07718472", "cucumber", "food", "vegetable"),
    (933, "n07697313", "cheeseburger", "food", "prepared_food"),
    (934, "n07697537", "hotdog", "food", "prepared_food"),
    (963, "n07873807", "pizza", "food", "prepared_food"),
]
TARGET_CLASSES = [
    {"label_id": a, "synset": b, "leaf_name": c, "level1": d, "level2": e}
    for a, b, c, d, e in TARGET_CLASSES
]

LEVEL1_NAMES = ["animal", "vehicle", "food"]
LEVEL2_NAMES = ["bird", "canine", "feline", "aircraft", "road_vehicle", "watercraft", "fruit", "vegetable", "prepared_food"]
LEAF_NAMES = [row["leaf_name"] for row in TARGET_CLASSES]
LEVEL1_TO_ID = {name: idx for idx, name in enumerate(LEVEL1_NAMES)}
LEVEL2_TO_ID = {name: idx for idx, name in enumerate(LEVEL2_NAMES)}
LEAF_TO_ID = {name: idx for idx, name in enumerate(LEAF_NAMES)}
LABEL_TO_RECORD = {row["label_id"]: row for row in TARGET_CLASSES}
LEVEL1_TO_LEVEL2 = {i: [] for i in range(len(LEVEL1_NAMES))}
LEVEL2_TO_LEAF = {i: [] for i in range(len(LEVEL2_NAMES))}
for leaf_id, row in enumerate(TARGET_CLASSES):
    l1 = LEVEL1_TO_ID[row["level1"]]
    l2 = LEVEL2_TO_ID[row["level2"]]
    if l2 not in LEVEL1_TO_LEVEL2[l1]:
        LEVEL1_TO_LEVEL2[l1].append(l2)
    LEVEL2_TO_LEAF[l2].append(leaf_id)

print(f"Level 1 groups: {len(LEVEL1_NAMES)} -> {LEVEL1_NAMES}")
print(f"Level 2 groups: {len(LEVEL2_NAMES)} -> {LEVEL2_NAMES}")
print(f"Leaf classes: {len(LEAF_NAMES)}")

Level 1 groups: 3 -> ['animal', 'vehicle', 'food']
Level 2 groups: 9 -> ['bird', 'canine', 'feline', 'aircraft', 'road_vehicle', 'watercraft', 'fruit', 'vegetable', 'prepared_food']
Leaf classes: 27


## 3. Optional standalone data preparation

The final results are embedded below, so data download is optional. To actually train from this single notebook, set `DOWNLOAD_DATA=True`. The downloader streams only the 27 target ImageNet classes from a Hugging Face mirror and writes local manifests.

In [3]:
DOWNLOAD_DATA = False
DATASET_REPO = "evanarlian/imagenet_1k_resized_256"
DATA_ROOT = Path("data/imagenet_27_food_standalone")
MAX_PER_CLASS = 20  # use None for full data
MANIFEST_FIELDS = ["split", "image_path", "filename", "synset", "label_id", "leaf_id", "leaf_name", "level1_id", "level1_name", "level2_id", "level2_name"]

def write_row(writer, split, filename, record):
    writer.writerow({
        "split": split,
        "image_path": str(Path("images") / split / record["synset"] / filename).replace("\\", "/"),
        "filename": filename,
        "synset": record["synset"],
        "label_id": record["label_id"],
        "leaf_id": LEAF_TO_ID[record["leaf_name"]],
        "leaf_name": record["leaf_name"],
        "level1_id": LEVEL1_TO_ID[record["level1"]],
        "level1_name": record["level1"],
        "level2_id": LEVEL2_TO_ID[record["level2"]],
        "level2_name": record["level2"],
    })

def download_split(split, max_per_class=20):
    from collections import defaultdict
    from datasets import Image as HFImage, load_dataset
    counts = defaultdict(int)
    targets = set(LABEL_TO_RECORD)
    manifest_dir = DATA_ROOT / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    with (manifest_dir / f"{split}.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        dataset = load_dataset(DATASET_REPO, split=split, streaming=True).cast_column("image", HFImage(decode=False))
        for sample in dataset:
            label_id = int(sample["label"])
            if label_id not in targets:
                continue
            if max_per_class is not None and counts[label_id] >= max_per_class:
                if all(counts[target] >= max_per_class for target in targets):
                    break
                continue
            record = LABEL_TO_RECORD[label_id]
            image_info = sample["image"]
            filename = Path(image_info["path"]).name
            out = DATA_ROOT / "images" / split / record["synset"] / filename
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists():
                out.write_bytes(image_info["bytes"])
            write_row(writer, split, filename, record)
            counts[label_id] += 1
    return dict(counts)

if DOWNLOAD_DATA:
    print("train", download_split("train", MAX_PER_CLASS))
    print("val", download_split("val", MAX_PER_CLASS))
else:
    print("Data download skipped. Set DOWNLOAD_DATA=True to train from this notebook.")

Data download skipped. Set DOWNLOAD_DATA=True to train from this notebook.


## 4. Dataset, model definitions, and training loops

These cells replace the repository code. They include flat CNN, compact Joint HEN, Coarse-to-Fine HEN, Common-Delta HEN, and a Classic HEN training recipe based on independent routers/experts.

In [4]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transforms(image_size=224, train=True):
    if train:
        return transforms.Compose([transforms.RandomResizedCrop(image_size, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2, 0.2, 0.2, 0.05), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([transforms.Resize(int(image_size * 1.15)), transforms.CenterCrop(image_size), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class ManifestDataset(Dataset):
    def __init__(self, manifest_path, image_size=224, train=True, mode="hier", level1_filter=None, level2_filter=None):
        self.manifest_path = Path(manifest_path)
        self.root = self.manifest_path.parent.parent
        self.mode = mode
        self.transform = build_transforms(image_size, train)
        self.rows = []
        with self.manifest_path.open("r", newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                if level1_filter and row["level1_name"] != level1_filter: continue
                if level2_filter and row["level2_name"] != level2_filter: continue
                self.rows.append(row)
        if not self.rows:
            raise ValueError(f"No rows in {manifest_path}")
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        with Image.open(self.root / row["image_path"]) as image:
            image = image.convert("RGB")
        image = self.transform(image)
        if self.mode == "flat": return image, int(row["leaf_id"])
        if self.mode == "level1": return image, int(row["level1_id"])
        if self.mode == "level2": return image, int(row["level2_id"])
        return image, int(row["level1_id"]), int(row["level2_id"]), int(row["leaf_id"])

print("Standalone dataset class ready.")

Standalone dataset class ready.


In [5]:
@dataclass
class HierOutput:
    level1_logits: torch.Tensor
    level1_log_probs: torch.Tensor
    level2_log_probs: torch.Tensor
    leaf_log_probs: torch.Tensor | None

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def make_feature_extractor(name="mobilenet_v3_large", pretrained=True):
    if name == "mobilenet_v3_large":
        m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None); return nn.Sequential(m.features, m.avgpool, nn.Flatten()), m.classifier[0].in_features
    if name == "mobilenet_v3_small":
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None); return nn.Sequential(m.features, m.avgpool, nn.Flatten()), m.classifier[0].in_features
    if name == "resnet18":
        m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None); return nn.Sequential(*(list(m.children())[:-1]), nn.Flatten()), m.fc.in_features
    if name == "convnext_tiny":
        m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None); return nn.Sequential(m.features, m.avgpool, nn.Flatten(1)), m.classifier[2].in_features
    if name == "shufflenet_v2_x1_0":
        m = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1 if pretrained else None); return nn.Sequential(m.conv1, m.maxpool, m.stage2, m.stage3, m.stage4, m.conv5, nn.AdaptiveAvgPool2d(1), nn.Flatten()), m.fc.in_features
    raise ValueError(name)

class FlatClassifier(nn.Module):
    def __init__(self, backbone="mobilenet_v3_small", num_classes=27, pretrained=True):
        super().__init__(); self.features, dim = make_feature_extractor(backbone, pretrained); self.head = nn.Linear(dim, num_classes)
    def forward(self, x): return self.head(self.features(x))

class JointHEN(nn.Module):
    def __init__(self, backbone="mobilenet_v3_large", pretrained=True, dropout=0.1):
        super().__init__(); self.features, dim = make_feature_extractor(backbone, pretrained); self.dropout = nn.Dropout(dropout)
        self.level1_head = nn.Linear(dim, 3)
        self.level2_heads = nn.ModuleDict({str(k): nn.Linear(dim, len(v)) for k, v in LEVEL1_TO_LEVEL2.items()})
        self.leaf_heads = nn.ModuleDict({str(k): nn.Linear(dim, len(v)) for k, v in LEVEL2_TO_LEAF.items()})
    def forward(self, x):
        f = self.dropout(self.features(x)); l1_logits = self.level1_head(f); l1 = F.log_softmax(l1_logits, dim=1)
        l2 = torch.full((x.size(0), 9), -1e9, device=x.device)
        for k, ids in LEVEL1_TO_LEVEL2.items(): l2[:, ids] = l1[:, k].unsqueeze(1) + F.log_softmax(self.level2_heads[str(k)](f), dim=1)
        leaf = torch.full((x.size(0), 27), -1e9, device=x.device)
        for k, ids in LEVEL2_TO_LEAF.items(): leaf[:, ids] = l2[:, k].unsqueeze(1) + F.log_softmax(self.leaf_heads[str(k)](f), dim=1)
        return HierOutput(l1_logits, l1, l2, leaf)

class CoarseToFineHEN(nn.Module):
    def __init__(self, router_size=128, expert_backbone="resnet18", pretrained=True):
        super().__init__(); self.router_size = router_size
        self.router_features, rdim = make_feature_extractor("shufflenet_v2_x1_0", pretrained); self.level1_head = nn.Linear(rdim, 3); self.level2_heads = nn.ModuleDict({str(k): nn.Linear(rdim, len(v)) for k, v in LEVEL1_TO_LEVEL2.items()})
        self.expert_features, edim = make_feature_extractor(expert_backbone, pretrained); self.leaf_heads = nn.ModuleDict({str(k): nn.Linear(edim, len(v)) for k, v in LEVEL2_TO_LEAF.items()})
    def forward(self, x, compute_leaf=True):
        r = F.interpolate(x, size=(self.router_size, self.router_size), mode="bilinear", align_corners=False); rf = self.router_features(r)
        l1_logits = self.level1_head(rf); l1 = F.log_softmax(l1_logits, dim=1); l2 = torch.full((x.size(0), 9), -1e9, device=x.device)
        for k, ids in LEVEL1_TO_LEVEL2.items(): l2[:, ids] = l1[:, k].unsqueeze(1) + F.log_softmax(self.level2_heads[str(k)](rf), dim=1)
        if not compute_leaf: return HierOutput(l1_logits, l1, l2, None)
        ef = self.expert_features(x); leaf = torch.full((x.size(0), 27), -1e9, device=x.device)
        for k, ids in LEVEL2_TO_LEAF.items(): leaf[:, ids] = l2[:, k].unsqueeze(1) + F.log_softmax(self.leaf_heads[str(k)](ef), dim=1)
        return HierOutput(l1_logits, l1, l2, leaf)

class CommonDeltaHEN(JointHEN):
    def __init__(self, backbone="mobilenet_v3_small", selected=("feline", "aircraft", "prepared_food"), delta_dim=32, pretrained=True):
        super().__init__(backbone, pretrained); _, dim = make_feature_extractor(backbone, False)
        self.delta = nn.ModuleDict({str(LEVEL2_TO_ID[name]): nn.Sequential(nn.Linear(dim, delta_dim), nn.ReLU(), nn.Linear(delta_dim, dim)) for name in selected})
    def forward(self, x):
        f = self.dropout(self.features(x)); l1_logits = self.level1_head(f); l1 = F.log_softmax(l1_logits, dim=1); l2 = torch.full((x.size(0), 9), -1e9, device=x.device)
        for k, ids in LEVEL1_TO_LEVEL2.items(): l2[:, ids] = l1[:, k].unsqueeze(1) + F.log_softmax(self.level2_heads[str(k)](f), dim=1)
        leaf = torch.full((x.size(0), 27), -1e9, device=x.device)
        for k, ids in LEVEL2_TO_LEAF.items():
            local = f + self.delta[str(k)](f) if str(k) in self.delta else f
            leaf[:, ids] = l2[:, k].unsqueeze(1) + F.log_softmax(self.leaf_heads[str(k)](local), dim=1)
        return HierOutput(l1_logits, l1, l2, leaf)

print("Standalone model definitions ready.")
print("Joint HEN MobileNetV3-Large params:", round(count_parameters(JointHEN(pretrained=False))/1e6, 2), "M")

Standalone model definitions ready.
Joint HEN MobileNetV3-Large params: 3.01 M


In [6]:
def run_flat_epoch(model, loader, optimizer=None):
    train = optimizer is not None; model.train(train); total = correct = loss_sum = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train: optimizer.zero_grad(set_to_none=True)
        logits = model(x); loss = F.cross_entropy(logits, y, label_smoothing=0.05)
        if train: loss.backward(); optimizer.step()
        loss_sum += loss.item() * x.size(0); correct += logits.argmax(1).eq(y).sum().item(); total += x.size(0)
    return {"loss": loss_sum/max(total,1), "acc": correct/max(total,1)}

def hier_loss(out, l1, l2, leaf):
    loss = 0.25 * F.cross_entropy(out.level1_logits, l1, label_smoothing=0.05) + 0.5 * F.nll_loss(out.level2_log_probs, l2)
    if out.leaf_log_probs is not None: loss = loss + F.nll_loss(out.leaf_log_probs, leaf)
    return loss

def run_hier_epoch(model, loader, optimizer=None):
    train = optimizer is not None; model.train(train); total = top = mid = leaf_ok = loss_sum = 0
    for x, l1, l2, leaf in loader:
        x, l1, l2, leaf = x.to(DEVICE), l1.to(DEVICE), l2.to(DEVICE), leaf.to(DEVICE)
        if train: optimizer.zero_grad(set_to_none=True)
        out = model(x); loss = hier_loss(out, l1, l2, leaf)
        if train: loss.backward(); optimizer.step()
        loss_sum += loss.item() * x.size(0); top += out.level1_logits.argmax(1).eq(l1).sum().item(); mid += out.level2_log_probs.argmax(1).eq(l2).sum().item(); leaf_ok += out.leaf_log_probs.argmax(1).eq(leaf).sum().item(); total += x.size(0)
    return {"loss": loss_sum/max(total,1), "top_acc": top/max(total,1), "mid_acc": mid/max(total,1), "leaf_acc": leaf_ok/max(total,1)}

def make_loaders(mode="hier", image_size=224, batch_size=96):
    train_ds = ManifestDataset(DATA_ROOT / "manifests/train.csv", image_size, True, mode)
    val_ds = ManifestDataset(DATA_ROOT / "manifests/val.csv", image_size, False, mode)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2), DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

def train_flat(backbone="mobilenet_v3_small", epochs=2, image_size=160, batch_size=64):
    train_loader, val_loader = make_loaders("flat", image_size, batch_size); model = FlatClassifier(backbone).to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4); best = 0
    for e in range(epochs):
        tr = run_flat_epoch(model, train_loader, opt); va = run_flat_epoch(model, val_loader); best = max(best, va["acc"]); print(e+1, tr, va, "best", best)
    return model, best

def train_hen(kind="joint", epochs=2, image_size=160, batch_size=64):
    train_loader, val_loader = make_loaders("hier", image_size, batch_size)
    model = {"joint": JointHEN, "c2f": CoarseToFineHEN, "common_delta": CommonDeltaHEN}[kind]().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4); best = 0
    for e in range(epochs):
        tr = run_hier_epoch(model, train_loader, opt); va = run_hier_epoch(model, val_loader); best = max(best, va["leaf_acc"]); print(e+1, tr, va, "best", best)
    return model, best

def classic_hen_training_recipe():
    return [
        "Train one top router on level1 labels: animal / vehicle / food.",
        "Train one mid router per level1 branch on its three level2 labels.",
        "Train one leaf expert per level2 branch on its three leaf labels.",
        "Inference runs one active path: top router -> selected mid router -> selected leaf expert.",
    ]

print("Training loops ready. They run only if RUN_LONG_TRAINING=True.")

Training loops ready. They run only if RUN_LONG_TRAINING=True.


## 5. Optional training entrypoint

This is the single switch for reproducing a selected experiment from inside the notebook. It is disabled by default to avoid accidentally launching hours of training.

In [7]:
EXPERIMENT = "joint_hen"  # flat_convnext, flat_mobilenet_small, joint_hen, c2f, common_delta

if RUN_LONG_TRAINING:
    if EXPERIMENT == "flat_convnext": model, best = train_flat("convnext_tiny", epochs=20, image_size=224, batch_size=96)
    elif EXPERIMENT == "flat_mobilenet_small": model, best = train_flat("mobilenet_v3_small", epochs=20, image_size=224, batch_size=96)
    elif EXPERIMENT == "joint_hen": model, best = train_hen("joint", epochs=25, image_size=224, batch_size=96)
    elif EXPERIMENT == "c2f": model, best = train_hen("c2f", epochs=12, image_size=224, batch_size=96)
    elif EXPERIMENT == "common_delta": model, best = train_hen("common_delta", epochs=25, image_size=224, batch_size=96)
    print("Best validation accuracy:", best)
else:
    print("Long training skipped. Set RUN_LONG_TRAINING=True and DOWNLOAD_DATA=True to reproduce a run.")
    print("Classic HEN recipe:")
    for line in classic_hen_training_recipe(): print("-", line)

Long training skipped. Set RUN_LONG_TRAINING=True and DOWNLOAD_DATA=True to reproduce a run.
Classic HEN recipe:
- Train one top router on level1 labels: animal / vehicle / food.
- Train one mid router per level1 branch on its three level2 labels.
- Train one leaf expert per level2 branch on its three leaf labels.
- Inference runs one active path: top router -> selected mid router -> selected leaf expert.


## 6. Final local results embedded from the report

The table below is copied from the local experiment summaries used by the final report. These are the values to inspect when training time is too expensive to repeat.

In [3]:
FINAL_RESULTS = [
    {
        "method": "Flat ConvNeXt-Tiny",
        "family": "Flat",
        "backbone": "ConvNeXt-Tiny",
        "accuracy_pct": 96.8889,
        "params_m": "27.84",
        "compute_gflops": "4.456",
        "status": "best absolute accuracy",
    },
    {
        "method": "Flat ResNet18 + SAM",
        "family": "Flat",
        "backbone": "ResNet18",
        "accuracy_pct": 94.9630,
        "params_m": "11.19",
        "compute_gflops": "1.814",
        "status": "close to 95%, below target",
    },
    {
        "method": "Flat MobileNetV3-Small",
        "family": "Flat",
        "backbone": "MobileNetV3-Small",
        "accuracy_pct": 94.5185,
        "params_m": "1.55",
        "compute_gflops": "0.057",
        "status": "too small to cross 95%",
    },
    {
        "method": "Classic 3-Level HEN",
        "family": "HEN classic",
        "backbone": "ResNet18 routers and experts",
        "accuracy_pct": 95.6296,
        "params_m": "145.31 total / 33.53 active",
        "compute_gflops": "5.442 active",
        "status": "accurate but storage-heavy",
    },
    {
        "method": "Coarse-to-Fine HEN full",
        "family": "HEN c2f",
        "backbone": "ShuffleNetV2 router + ResNet18",
        "accuracy_pct": 92.3704,
        "params_m": "36.23",
        "compute_gflops": "~0.64 active estimate",
        "status": "did not reach leaf target",
    },
    {
        "method": "Joint HEN MobileNetV3-Large",
        "family": "HEN compact",
        "backbone": "MobileNetV3-Large",
        "accuracy_pct": 95.6296,
        "params_m": "3.01",
        "compute_gflops": "0.217",
        "status": "best efficiency tradeoff",
    },
    {
        "method": "Selective Common-Delta HEN",
        "family": "HEN research",
        "backbone": "MobileNetV3-Small",
        "accuracy_pct": 93.7778,
        "params_m": "1.74 total / 0.79 trained",
        "compute_gflops": "0.057 backbone-only",
        "status": "useful locally, not winner",
    },
]

columns = ["method", "family", "backbone", "accuracy_pct", "params_m", "compute_gflops", "status"]
widths = {column: max(len(column), *(len(str(row[column])) for row in FINAL_RESULTS)) for column in columns}
for column in columns:
    print(str(column).ljust(widths[column]), end="  ")
print()
for row in FINAL_RESULTS:
    for column in columns:
        value = f"{row[column]:.4f}" if isinstance(row[column], float) else str(row[column])
        print(value.ljust(widths[column]), end="  ")
    print()

method                              family        backbone                         accuracy_pct  params_m                       compute_gflops            status
Flat ConvNeXt-Tiny                  Flat          ConvNeXt-Tiny                    96.8889       27.84                          4.456                    best absolute accuracy
Flat ResNet18 + SAM                 Flat          ResNet18                         94.9630       11.19                          1.814                    close to 95%, below target
Flat MobileNetV3-Small              Flat          MobileNetV3-Small                94.5185       1.55                           0.057                    too small to cross 95%
Classic 3-Level HEN                 HEN classic   ResNet18 routers and experts     95.6296       145.31 total / 33.53 active    5.442 active             accurate but storage-heavy
Coarse-to-Fine HEN full             HEN c2f       ShuffleNetV2 router + ResNet18   92.3704       36.23                         

## 7. Resource-efficiency calculation

The main comparison is not whether HEN beats the strongest flat CNN in absolute accuracy. It does not. The key result is that compact Joint HEN crosses the practical 95% line with a much smaller resource budget.

In [4]:
flat_acc = 96.8889
flat_params = 27.84
flat_gflops = 4.456
hen_acc = 95.6296
hen_params = 3.01
hen_gflops = 0.217

print("Flat ConvNeXt-Tiny vs Compact Joint HEN:")
print(f"- Accuracy gap: {flat_acc - hen_acc:.2f} percentage points")
print(f"- Parameter ratio: {flat_params / hen_params:.2f}x more parameters for flat ConvNeXt-Tiny")
print(f"- Compute ratio: {flat_gflops / hen_gflops:.2f}x more GFLOPs for flat ConvNeXt-Tiny")
print(
    f"- Compact HEN keeps {hen_acc / flat_acc * 100:.2f}% of the flat ConvNeXt accuracy "
    f"using {hen_params / flat_params * 100:.2f}% of the parameters and "
    f"{hen_gflops / flat_gflops * 100:.2f}% of the compute."
)

Flat ConvNeXt-Tiny vs Compact Joint HEN:
- Accuracy gap: 1.26 percentage points
- Parameter ratio: 9.25x more parameters for flat ConvNeXt-Tiny
- Compute ratio: 20.53x more GFLOPs for flat ConvNeXt-Tiny
- Compact HEN keeps 98.70% of the flat ConvNeXt accuracy using 10.81% of the parameters and 4.87% of the compute.


In [ ]:
# Optional visualization. Run this cell to regenerate the accuracy/parameter/compute chart.
import matplotlib.pyplot as plt

plot_rows = [
    ("Flat ConvNeXt", 96.8889, 27.84, 4.456),
    ("Classic HEN", 95.6296, 145.31, 5.442),
    ("Compact Joint HEN", 95.6296, 3.01, 0.217),
    ("Common-Delta", 93.7778, 1.74, 0.057),
]
labels = [row[0] for row in plot_rows]
accuracy = [row[1] for row in plot_rows]
params = [row[2] for row in plot_rows]
compute = [row[3] for row in plot_rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(labels, accuracy)
axes[0].axhline(95, color="tab:red", linestyle="--", linewidth=1)
axes[0].set_title("Accuracy (%)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(labels, params)
axes[1].set_title("Parameters (M)")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(labels, compute)
axes[2].set_title("Compute (GFLOPs)")
axes[2].tick_params(axis="x", rotation=25)

fig.tight_layout()
plt.show()

## 10. Final interpretation

- The strongest absolute model remains **Flat ConvNeXt-Tiny** at **96.89%**.
- The best compact answer is **Joint HEN + MobileNetV3-Large** at **95.63%**, using **3.01M parameters** and about **0.217 GFLOPs**.
- The 95% target is practical because the validation set contains ambiguous mixed-object images, so the last percentage point is partly architecture and partly data-label noise.
- Coarse-to-Fine and Common-Delta were useful research directions, but neither became the final efficiency/accuracy winner.